# API có phân trang, rate limit và incremental: động đất toàn cầu 2024 (USGS)

Bài basic 01 gọi API một lần là xong. API thật thường không cho vậy: kết quả trả **từng trang**, bạn
phải **tự giới hạn tốc độ** để không bị chặn, và quan trọng nhất, **bản ghi cũ bị sửa**. Lấy lại
toàn bộ mỗi đêm thì tốn; chỉ lấy "bản ghi mới theo thời gian xảy ra" thì bỏ sót bản sửa.

API: [USGS FDSN Event Web Service](https://earthquake.usgs.gov/fdsnws/event/1/), dữ liệu công (public
domain), không cần key. Mốc ghim: động đất **M ≥ 4.5 toàn cầu**, `starttime=2024-01-01`,
`endtime=2025-01-01` (~6 400 trận), phân trang bằng `limit` / `offset` (**offset bắt đầu từ 1**),
sắp `orderby=time-asc`. Mỗi bản ghi có `time` (lúc xảy ra) và `updated` (lần sửa cuối, mili giây epoch).

Dữ liệu thô: `data/raw/usgs-earthquakes/run-NNN/page-NNN.json`, trạng thái incremental ở
`data/raw/usgs-earthquakes/state.json`. Test không gọi mạng (dùng server giả). Chấm:
`uv run pytest tracks/data-engineer/modules/00-introduction/exercises/intermediate/01-paginated-incremental-api -q`

In [ ]:
# explore: tìm gốc repo; dữ liệu thô của bài này do chính bạn thu thập
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw" / "usgs-earthquakes"
CLEAN = ROOT / "data" / "clean" / "usgs-earthquakes"
print(RAW)

In [ ]:
import json
import time
from collections.abc import Iterator
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

API_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
START, END, MIN_MAG = "2024-01-01", "2025-01-01", 4.5
PAGE_SIZE = 1000


def fetch_json(get, url: str, params: dict, max_retries: int = 4, sleep=time.sleep) -> dict:
    """Đã viết sẵn (chính là bài basic 01): timeout, retry 429/5xx/lỗi mạng với backoff hoặc Retry-After."""
    for attempt in range(max_retries + 1):
        wait = 2 ** attempt
        try:
            resp = get(url, params=params, timeout=60)
        except requests.RequestException as exc:
            error = f"lỗi mạng: {exc}"
        else:
            if resp.status_code == 200:
                return resp.json()
            if resp.status_code != 429 and resp.status_code < 500:
                raise RuntimeError(f"HTTP {resp.status_code}")
            error = f"HTTP {resp.status_code}"
            wait = int(resp.headers.get("Retry-After", wait))
        if attempt < max_retries:
            sleep(wait)
    raise RuntimeError(f"Hết {max_retries} lần thử lại, lỗi cuối: {error}")

## 1. Tham số một trang

`page_params(start, end, min_mag, limit, offset, updated_after=None) -> dict` với các khoá `format`
(`"geojson"`), `starttime`, `endtime`, `minmagnitude`, `orderby` (`"time-asc"`), `limit`, `offset`.
Nếu có `updated_after` (mili giây epoch, UTC) thì thêm `updatedafter` dạng `"YYYY-MM-DDTHH:MM:SS.mmm"` (UTC, 3 chữ số mili giây).

In [ ]:
def page_params(start: str, end: str, min_mag: float, limit: int, offset: int,
                updated_after: int | None = None) -> dict:
    """Tham số query cho một trang; updated_after (ms epoch) -> 'updatedafter'."""
    # TODO
    raise NotImplementedError

## 2. Duyệt các trang, tự giới hạn tốc độ

`fetch_pages(get, params, page_size, min_interval=1.0, sleep=time.sleep, clock=time.monotonic)` là một
**generator** trả lần lượt payload từng trang:

- trang đầu dùng `params` như được truyền vào (`offset` của nó), trang sau tăng `offset` thêm `page_size`;
- gọi qua `fetch_json(get, API_URL, ...)` (đã có retry);
- **dừng** khi trang có ít hơn `page_size` bản ghi (kể cả 0). Nếu tổng số bản ghi chia hết cho
  `page_size`, bạn sẽ gọi thêm một trang rỗng: chấp nhận được;
- **throttle**: giữa hai request liên tiếp phải cách nhau ít nhất `min_interval` giây, tính bằng `clock()`;
  thiếu bao nhiêu thì `sleep` bấy nhiêu. Không nghỉ trước request đầu tiên.

In [ ]:
def fetch_pages(get, params: dict, page_size: int, min_interval: float = 1.0,
                sleep=time.sleep, clock=time.monotonic) -> Iterator[dict]:
    """Generator: payload từng trang; offset += page_size; dừng ở trang ngắn; cách nhau >= min_interval giây."""
    # TODO
    raise NotImplementedError

## 3. Thu thập incremental theo `updated`

`collect(get, raw_dir, start=START, end=END, min_mag=MIN_MAG, page_size=PAGE_SIZE, min_interval=1.0,
sleep=time.sleep, clock=time.monotonic) -> dict`:

1. đọc `raw_dir/state.json` nếu có: `{"watermark": <ms>, "runs": <số lần chạy thành công>}`;
2. lần chạy này là `run = runs + 1`; trang thứ k ghi **nguyên payload** vào `raw_dir/run-{run:03d}/page-{k:03d}.json` (k từ 1);
3. lần đầu (chưa có state) lấy toàn bộ; lần sau thêm `updated_after=watermark`: chỉ những bản ghi **mới hoặc
   bị sửa** từ lần trước, dù chúng xảy ra từ tháng 1;
4. **chỉ khi mọi trang đã ghi xong** mới cập nhật `state.json` (ghi qua tệp tạm): watermark mới là `updated`
   lớn nhất đã thấy (không nhỏ hơn watermark cũ). Lỗi giữa chừng thì state giữ nguyên, lần sau chạy lại đúng lượt đó;
5. trả `{"run": run, "pages": số trang, "events": số bản ghi đã lấy, "watermark": watermark mới}`.

`updatedafter` của USGS tính cả mốc bằng, nên bản ghi tại đúng watermark bị lấy lại: **at-least-once**,
và bước `load_events` phải dedup (upsert) theo `id`.

In [ ]:
def collect(get, raw_dir: Path, start: str = START, end: str = END, min_mag: float = MIN_MAG,
            page_size: int = PAGE_SIZE, min_interval: float = 1.0,
            sleep=time.sleep, clock=time.monotonic) -> dict:
    """Lần đầu lấy hết, lần sau chỉ lấy updated >= watermark; ghi trang thô; state chỉ cập nhật khi xong."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: thu thập (lần đầu ~7 request cách nhau 1 giây; chạy lại cell = một lượt incremental)
summary = collect(requests.get, RAW)
summary

## 4. Gộp các lượt thành bảng sạch (upsert)

`load_events(raw_dir)` đọc mọi `run-*/page-*.json`, mỗi feature một dòng với đúng các cột:

`id`, `time`, `updated` (datetime **UTC**), `mag`, `mag_type`, `place`, `region`, `lat`, `lon`,
`depth_km` (từ `geometry.coordinates` = `[lon, lat, depth]`), `tsunami` (bool), `status`, `sig`, `type`.

- `region`: phần sau dấu `", "` cuối cùng của `place` (cả `place` nếu không có dấu phẩy), bỏ đuôi `" Earthquake"`;
- cùng `id` xuất hiện ở nhiều lượt: giữ bản có `updated` lớn nhất (upsert);
- sắp theo `time`, đánh lại index.

In [ ]:
COLUMNS = ["id", "time", "updated", "mag", "mag_type", "place", "region",
           "lat", "lon", "depth_km", "tsunami", "status", "sig", "type"]


def load_events(raw_dir: Path) -> pd.DataFrame:
    """Gộp mọi run-*/page-*.json; một dòng mỗi id (bản updated mới nhất); cột theo COLUMNS."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: bảng sạch
df = load_events(RAW)
print(len(df), "trận,", df["id"].is_unique)
df.head()

## 5. Insight

`insights(df) -> dict`:

| Khoá | Nghĩa |
| --- | --- |
| `events`, `m6_plus`, `m7_plus` | số trận, số trận M ≥ 6, M ≥ 7 |
| `largest_mag`, `largest_place`, `largest_time` | trận lớn nhất: magnitude, `place`, ngày UTC `"YYYY-MM-DD"` |
| `top_region`, `top_region_events` | `region` nhiều trận nhất và số trận |
| `busiest_month`, `busiest_month_events` | tháng (1–12, UTC) nhiều trận nhất |
| `tsunami_flagged` | số trận có cờ `tsunami` |
| `median_update_lag_days` | trung vị của `updated - time`, tính bằng ngày (số thực) |
| `share_updated_after_30d` | tỉ lệ bản ghi có `updated - time` > 30 ngày |
| `gutenberg_richter_b` | hệ số b của luật Gutenberg–Richter (xem dưới) |

Luật Gutenberg–Richter: số trận có magnitude ≥ M giảm theo luỹ thừa,

$$\log_{10} N(\ge M) = a - b\,M$$

Với các ngưỡng M = 4.5, 5.0, ..., 7.0 (`np.arange(4.5, 7.01, 0.5)`) có N > 0, fit đường thẳng
`np.polyfit(M, log10(N), 1)`; b là **trừ** hệ số góc.

In [ ]:
def insights(df: pd.DataFrame) -> dict:
    """Các con số trong bảng trên."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: các con số để viết report.md
INSIGHTS = insights(df)
INSIGHTS

## 6. Biểu đồ

Ít nhất hai biểu đồ, mỗi cái một cell `explore`. Gợi ý:

1. bản đồ phân tán `lon`/`lat`, kích thước theo magnitude: bạn sẽ thấy "Vành đai lửa Thái Bình Dương" mà không cần bản đồ nền;
2. số trận theo tuần: tuần nào vọt lên, sau trận nào?
3. `N(≥M)` theo M trên trục log: đường gần thẳng là luật Gutenberg–Richter;
4. phân phối `updated - time` (ngày): vì sao incremental theo `time` là sai với nguồn này?

In [ ]:
# explore: bản đồ phân tán
# TODO: scatter lon/lat, kích thước theo mag, màu theo depth_km

In [ ]:
# explore: số trận theo tuần
# TODO: số trận mỗi tuần (resample "W"), đánh dấu vài trận lớn nhất

In [ ]:
# explore: (tuỳ chọn) Gutenberg-Richter, độ trễ cập nhật

## 7. Ghi dữ liệu sạch

`write_clean(df, out_dir)` ghi đè `out_dir/events.parquet` (qua tệp tạm) và trả đường dẫn.

In [ ]:
def write_clean(df: pd.DataFrame, out_dir: Path) -> Path:
    """Ghi đè out_dir/events.parquet (qua tệp tạm), trả đường dẫn."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: ghi data/clean/usgs-earthquakes/events.parquet
path = write_clean(df, CLEAN)
print(path, len(pd.read_parquet(path)), "dòng")

## 8. Báo cáo

Viết `report.md` cạnh notebook: `## Bộ dữ liệu`, `## Làm sạch`, `## Insight` (có số), `## Giới hạn`.
Bắt buộc trả lời: nếu chạy pipeline này mỗi đêm, chọn watermark theo `time` hay `updated`, vì sao, và
mỗi đêm tốn bao nhiêu request?